<a href="https://colab.research.google.com/github/FelipeOliveira948/Headache-Musical/blob/main/desafio1/Desafio1_API_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Célula 01 · Instalar**

In [1]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker

**Célula 02 · Importações e Autenticação (service account)**

In [2]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@storied-box-513817-n2.iam.gserviceaccount.com


**Célula 03 · Abrir a planilha**

In [15]:
# Informe o ID da planilha e o nome que será dado à aba.
from google.colab import userdata
SHEET_ID = userdata.get("Planilha")
NOME_TABELA = "atracoes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Festival_Musica | atracoes | https://docs.google.com/spreadsheets/d/1NJuDpqD2si5dq4EpePmp_WPVRFuOcXjzHOojqgj6Ydg


**Célula 04 · Cabeçalho – Tabela Dimensão**

In [16]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["atracao_id", "nome_artista_ficticio",
           "genero_musical", "edicao_festival", "dia_apresentacao",
           "palco", "horario_inicio" ]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


**Célula 05 · 10 linhas de teste – Tabela Dimensão**

In [17]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    NOMES_ARTISTA = ["Endset", "Skullet", "Kiwi Assasino", "Tetako Music","Gissaroll Gogh", "Mel do Amor", "Dejavoodoo"]
    GENERO = ["Rock", "Eletronica", "Bossa Nova", "Afrobeat", "Sertanejo","Pop", "Rap"]
    FESTIVAL = ["Festival Multi Music 26", "Festival Mar Vermelho 26", "Festival Ovo Cozido 26"]
    # Cria um registro fictício para as colunas definidas no esquema.
    # Personalize de acordo com o seu esquema de dados
    return {
        "atracao_id": i,
        "nome_artista_ficticio": random.choice(NOMES_ARTISTA),
        "genero_musical":random.choice(GENERO),
        "edicao_festival": random.choice(FESTIVAL),
        "dia_apresentacao": fake.date_this_year().isoformat(),   # datas como texto
        "palco": random.choice(["principal" , "secundario" , "eletronico" , "acustico"]),
        "horario_inicio": random.choice(["16:00", "18:00", "17:00"])

    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


**CÉLULA 6 · Conferência – Tabela Dimensão**

In [18]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 21
Cabeçalho confere: True
[['atracao_id', 'nome_artista_ficticio', 'genero_musical', 'edicao_festival', 'dia_apresentacao', 'palco', 'horario_inicio'], ['1', 'Mel do Amor', 'Rock', 'Festival Multi Music 26', '2026-07-16', 'eletronico', '16:00'], ['2', 'Skullet', 'Eletronica', 'Festival Ovo Cozido 26', '2026-01-08', 'principal', '17:00']]


**CÉLULA 7 · Funções de envio de dados - Planilha**

In [19]:
# Importa recursos para controlar pausas, acessar a planilha e tratar tabelas.
import time
import gspread
import pandas as pd

def df_para_valores(df):
    """DataFrame -> lista de listas com tipos nativos do Python."""
    # Trabalha em uma cópia para não alterar o DataFrame original.
    df = df.copy()
    for c in df.columns:
        # Converte datas para texto em um formato padronizado.
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d %H:%M:%S")
        # Converte outros valores de data e hora para texto.
        elif df[c].dtype == object:
            df[c] = df[c].map(
                lambda v: v.isoformat() if hasattr(v, "isoformat") else v)
    # Substitui valores ausentes por texto vazio e retorna linhas de valores.
    return df.astype(object).where(df.notna(), "").values.tolist()

def enviar(sh, df, nome_aba, lote=5000, pausa=2):
    """Escreve cabeçalho e linhas numa aba, em lotes, e confere a contagem."""
    # Formata números com ponto como separador de milhar.
    n = lambda x: f"{x:,}".replace(",", ".")
    try:
        # Reutiliza a aba existente e apaga seu conteúdo anterior.
        ws = sh.worksheet(nome_aba)
        ws.clear()
    except gspread.WorksheetNotFound:
        # Cria uma aba caso ela ainda não exista.
        ws = sh.add_worksheet(title=nome_aba, rows=len(df) + 1,
                              cols=len(df.columns))
    # Ajusta o tamanho da aba e grava os nomes das colunas.
    ws.resize(rows=len(df) + 1, cols=len(df.columns))
    ws.update(range_name="A1", values=[list(df.columns)])
    # Converte o DataFrame e envia as linhas em lotes.
    valores = df_para_valores(df)
    for i in range(0, len(valores), lote):
        ws.append_rows(valores[i:i + lote], value_input_option="RAW")
        print(f"  {nome_aba}: {n(min(i + lote, len(valores)))} / {n(len(valores))}")
        time.sleep(pausa)               # respeita o limite de escritas/minuto
    # Confere se a quantidade de linhas gravadas corresponde à esperada.
    lidas = len(ws.col_values(1)) - 1   # lê de volta e confere
    if lidas != len(df):
        raise RuntimeError(f"Aba {nome_aba}: enviei {len(df)}, li {lidas}.")
    print(f"OK {nome_aba}: {n(len(df))} linhas na planilha")

def remover_aba_padrao(sh):
    """Apaga a aba vazia que o Google cria junto com a planilha nova."""
    # Localiza e remove a aba padrão somente se houver outra aba disponível.
    abas = sh.worksheets()
    padrao = ("Página1", "Sheet1", "Planilha1")
    for w in abas:
        if w.title in padrao and len(abas) > 1 and not w.get_all_values():
            sh.del_worksheet(w)

**CÉLULA 8 · Carregar Bases de dados - Prontas**

In [26]:
# Abre a opção de selecionar arquivos CSV no Colab e envia os arquivos.
from google.colab import files
files.upload()   # selecione os arquivos CSV do seu tema

# Carrega cada CSV em um DataFrame e associa-o ao nome da tabela.
# Altere o nome da tabela de acordo com a sua Base de dados
BASES = {
    "atracoes": pd.read_csv("atracoes.csv"),
    "ingressos_vendidos": pd.read_csv("ingressos_vendidos.csv"),
}
# Calcula e exibe a quantidade de linhas carregadas.
total = sum(len(d) for d in BASES.values())
print({n: len(d) for n, d in BASES.items()}, "| total:", total)

Saving ingressos_vendidos.csv to ingressos_vendidos (1).csv
Saving atracoes.csv to atracoes.csv
{'atracoes': 10100, 'ingressos_vendidos': 55000} | total: 65100


**CÉLULA 9 · Enviar**

In [29]:
# Envia cada tabela para uma aba e remove a aba padrão, se estiver vazia.
for nome, df in BASES.items():
    enviar(sh, df, nome)        # uma aba por tabela
remover_aba_padrao(sh)

  atracoes: 5.000 / 10.100
  atracoes: 10.000 / 10.100
  atracoes: 10.100 / 10.100
OK atracoes: 10.100 linhas na planilha
  ingressos_vendidos: 5.000 / 55.000
  ingressos_vendidos: 10.000 / 55.000
  ingressos_vendidos: 15.000 / 55.000
  ingressos_vendidos: 20.000 / 55.000
  ingressos_vendidos: 25.000 / 55.000
  ingressos_vendidos: 30.000 / 55.000
  ingressos_vendidos: 35.000 / 55.000
  ingressos_vendidos: 40.000 / 55.000
  ingressos_vendidos: 45.000 / 55.000
  ingressos_vendidos: 50.000 / 55.000
  ingressos_vendidos: 55.000 / 55.000
OK ingressos_vendidos: 55.000 linhas na planilha


**CÉLULA 10 · Validar**

In [30]:
# Soma e exibe a quantidade total de linhas das tabelas.
total = sum(len(d) for d in BASES.values())
print("Total de linhas:", f"{total:,}".replace(",", "."))
# Interrompe a execução se o volume mínimo não for atingido.
assert total >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

# Confere os cabeçalhos e a quantidade de registros em cada aba.
for nome, df in BASES.items():
    w = sh.worksheet(nome)
    assert w.row_values(1) == list(df.columns), f"Cabeçalho diferente em {nome}"
    assert len(w.col_values(1)) - 1 == len(df), f"Contagem diferente em {nome}"
# Confirma que todas as verificações passaram.
print("A planilha confere com a base gerada.")

Total de linhas: 65.100
A planilha confere com a base gerada.
